# 0. Problem
## 1327. List the Products Ordered in a Period — Easy
For February 2020, return products whose total ordered units are at least 100.

Official: https://leetcode.com/problems/list-the-products-ordered-in-a-period/

# 1. Setup

In [ ]:
import pandas as pd
products_rows=[(1,"Leetcode Solutions","Book"),(2,"Jewels of Stringology","Book"),(3,"HP","Laptop"),(4,"Lenovo","Laptop"),(5,"Leetcode Kit","T-shirt")]
orders_rows=[(1,"2020-02-05",60),(1,"2020-02-10",70),(2,"2020-01-18",30),(2,"2020-02-11",80),(3,"2020-02-17",2),(3,"2020-02-24",3),(4,"2020-03-01",20),(5,"2020-02-25",50),(5,"2020-02-27",50)]
products_pd=pd.DataFrame(products_rows,columns=["product_id","product_name","product_category"])
orders_pd=pd.DataFrame(orders_rows,columns=["product_id","order_date","unit"])
orders_pd["order_date"]=pd.to_datetime(orders_pd["order_date"])
products_pd,orders_pd

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
spark=SparkSession.builder.getOrCreate()
products_spark=spark.createDataFrame(products_rows,["product_id","product_name","product_category"])
orders_spark=spark.createDataFrame(orders_rows,["product_id","order_date","unit"]).withColumn("order_date",F.to_date("order_date"))
products_spark.createOrReplaceTempView("Products")
orders_spark.createOrReplaceTempView("Orders")

# 2. SQL Solution

In [ ]:
sql_result=spark.sql("""
SELECT p.product_name,SUM(o.unit) AS unit
FROM Products p
JOIN Orders o ON p.product_id=o.product_id
WHERE o.order_date BETWEEN DATE('2020-02-01') AND DATE('2020-02-29')
GROUP BY p.product_id,p.product_name
HAVING SUM(o.unit)>=100
ORDER BY p.product_name
""")
sql_result.show(truncate=False)

# 3. pandas Solution

In [ ]:
feb=orders_pd.loc[orders_pd["order_date"].between(pd.Timestamp("2020-02-01"),pd.Timestamp("2020-02-29"))]
totals=feb.groupby("product_id",as_index=False).agg(unit=("unit","sum"))
result_pd=(totals.loc[totals["unit"]>=100].merge(products_pd[["product_id","product_name"]],on="product_id")[["product_name","unit"]].sort_values("product_name").reset_index(drop=True))
result_pd

# 4. PySpark Solution

In [ ]:
feb=(orders_spark.filter(F.col("order_date").between(F.to_date(F.lit("2020-02-01")),F.to_date(F.lit("2020-02-29")))).groupBy("product_id").agg(F.sum("unit").alias("unit")).filter(F.col("unit")>=100))
result_spark=feb.join(products_spark,on="product_id").select("product_name","unit").orderBy("product_name")
result_spark.show(truncate=False)

# 5. Pattern Mapping
| Concept | SQL | pandas | PySpark |
|---|---|---|---|
| month filter | date `BETWEEN` | `.between()` | `.between()` |
| group threshold | `HAVING SUM>=100` | aggregate then filter | aggregate then filter |

# 6. Muscle-Memory Round

พิมพ์ใหม่เองโดยไม่ copy คำตอบด้านบน

In [ ]:
# MUSCLE MEMORY — SQL
# Rebuild using temp view(s): Products, Orders

In [ ]:
# MUSCLE MEMORY — PANDAS
# Rebuild using: products_pd, orders_pd

In [ ]:
# MUSCLE MEMORY — PYSPARK
# Rebuild using: products_spark, orders_spark